## __Predicting Airbnb Listing Prices in Melbourne, Australia__
---

**Goal:** Predict listed prices of Airbnb properties in Melbourne based on various Airbnb characteristics and regression models 

**Problem**: Currently there is no convenient way for a new Airbnb host to decide the price of his or her listing. New hosts must often rely on the price of neighbouting listings when deciding on the price of their own listing. 

The project aims at predicting Airbnb listing prices for Melbourne based on the characteristics of listed properties to help hosts decide reasonable price. A Predictive Price Modelling tool whereby a new host can enter all the relevant details such as location of the listing, listing properties, available amenities etc and the Machine Learning Model will suggest the Price for the listing. We will train and make comparisons between different methods. 

## Problem Description and Initial Data Analysis

In the project, I will employ MSE to evaluate the performance of the linear regression model, Decision Trees and Random Forests. The Mean Squared Error (MSE) is the average of the summation of the squared difference between the estimated values and actual values. It is an important to reduce the value of MSE in evaluating the performance of machine learning models.

$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$

In [2]:
# import libraries 
import pandas as pd 
import numpy as np 

import seaborn as sns 
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure 

**Forcasting Problem**

This project aims to develop a machine learning model to predict the housing price of Airbnb listing in Melbourne based on property features. The goal is to train the model using a labelled dataset and apply it to predict prices for unseen listings with high accuracy.

****Evaluation criteria****

This project evaluates model accuracy using **Mean Absolute Error (MAE)**, which measures the average absolute difference between the predicted and actual prices. The formulas as below:

<div align="center">
  <img src="MAE_formula.png" alt="MAE Formula" width="300"/>
</div>

- **Advantages:**
    - Easy to interpret: provides a clear, dollar-based measure of prediction error, making it accessible to non-technical stakeholders.
    - Consistent scaling: treat all errors equally, regardless of the property's price.
    - Reduced sensitivity to outliers: avoid being overly influenced by a small number of unusually high-priced listings.

- **Limitations:**
    - Equal weighting of all errors: weight large error on a premium listing the same as a similar-sized error on a lower-segment property, even though the impact may differ.
    - No relative context: does not reflect the size of the error relative to the actual price.

****Categorise Variables****

| Variable Type | Number of Features | Feature Names |
|:-------------:|:-----------------:|:--------------|
| Numerical     |        37         | host_response_rate<br>host_acceptance_rate<br>host_listings_count<br>latitude<br>longitude<br>accommodates<br>bathrooms<br>bedrooms<br>beds<br>minimum_nights<br>maximum_nights<br>minimum_minimum_nights<br>maximum_minimum_nights<br>minimum_maximum_nights<br>maximum_maximum_nights<br>minimum_nights_avg_ntm<br>maximum_nights_avg_ntm<br>availability_30<br>availability_60<br>availability_90<br>availability_365<br>number_of_reviews<br>number_of_reviews_ltm<br>number_of_reviews_l30d<br>review_scores_rating<br>review_scores_accuracy<br>review_scores_cleanliness<br>review_scores_checkin<br>review_scores_communication<br>review_scores_location<br>review_scores_value<br>calculated_host_listings_count<br>calculated_host_listings_count_entire_homes<br>calculated_host_listings_count_private_rooms<br>calculated_host_listings_count_shared_rooms<br>reviews_per_month |
| Ordinal       |         2         | host_response_time<br>room_type |
| Nominal       |         7         | host_is_superhost<br>host_verifications<br>host_has_profile_pic<br>host_identity_verified<br>property_type<br>has_availability<br>instant_bookable |
| Text          |        10         | source<br>name<br>description<br>neighborhood_overview<br>host_name<br>host_location<br>host_about<br>host_neighbourhood<br>neighbourhood<br>neighbourhood_cleansed<br>amenities |
| Date          |         3         | host_since<br>first_review<br>last_review |


> **Note**: The variables `ID` and `price` are not included in the table above. `ID` is a unique identifier, while `price` is the target variable and is treated separately. Although `price` appears as text in its raw form (eg. $132.00), it will be converted to a numerical type for modeling purposes.

### Missing Values 

In [3]:
# Load train and test sets
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Summarise missing values
def missing_summary(df, name):
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    percent = (missing / len(df)) * 100
    return pd.DataFrame({
        f'{name}_Missing': missing,
        f'{name}_Missing(%)': percent.round(2)
    })

# Generate missing value summaries
train_missing = missing_summary(train, "Train")
test_missing = missing_summary(test, "Test")

# Combine summaries
missing_df = pd.concat([train_missing, test_missing], axis=1)

# Drop rows with all missing values (feature not missing in either set)
missing_df = missing_df.dropna(how="all")

# Display
missing_df.style.set_caption("Missing Values Summary").format(precision=2)

,Train_Missing,Train_Missing(%),Test_Missing,Test_Missing(%)
host_neighbourhood,3341.00,47.73,2213.00,73.77
host_about,2280.00,32.57,1451.00,48.37
neighbourhood,1791.00,25.59,1470.00,49.00
neighborhood_overview,1791.00,25.59,1470.00,49.00
host_location,1356.00,19.37,720.00,24.00
bedrooms,358.00,5.11,82.00,2.73
room_type,134.00,1.91,41.00,1.37
neighbourhood_cleansed,108.00,1.54,42.00,1.40
beds,85.00,1.21,12.00,0.40
property_type,84.00,1.20,39.00,1.30


The dataset contains several features with missing values in both the training and test sets, though the degree of missingness varies.

- **High missing rate**: The ***location-related field*** shows substantial missingness. `host_neighbourhood` is missing in nearly half of the training set (49.43%) and over two-thirds of the test set (68.87%). Particularly, `neighbourhood` and `neighborhood_overview` show more than 47% in test set.

- **Moderate missing rate**: Variables such as `bedrooms`, and `host_acceptance_rate` exhibit moderate levels of missingness, ranging from 0.9% to around 25%, with `host_location` missing in about 18.29% of the training set and 25.67% of the test set.

- **Low missing rate**: Most numerical and review-based features have low missing (under 1% in train set and 10% in test set), making them manageable for imputation or cleaning.

****Univariate data characteristics in training set****

---

**Exploring target variable (`price`)**
- In this part, I applied a logarithmic transformation to the price variable. The original price values exhibit a highly skewed and diverse range. Applying the log transformation helps normalise the distribution, making it easier to identify underlying trends and outliers in the data.

***Price Distribution***:

In [4]:
# Transform 'price' to numeric
train['price'] = train['price'].str.replace(r'[\$,]', '', regex=True).astype(float)
# Log-transform price (use log1p to handle extreme values for visualisaion)
train['price_log'] = np.log1p(train['price'])

# Summary table
summary = train['price'].describe()
summary_table = pd.DataFrame(summary).T.round(2)
display(summary_table)

# Price histogram
plt.figure(figsize=(10, 6))a
sns.histplot(train['price_log'], bins=50, kde=True, color="#4CB06A", edgecolor='white')
plt.title('Price Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Price (log)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Log-transformed prices
log_prices = [4, 5, 6]
original_prices = np.expm1(log_prices)

# Create a table
price_table = pd.DataFrame({
    'Price(log)': log_prices,
    'Original Price': original_prices.round(2)
})

display(price_table)

SyntaxError: invalid syntax (3417743940.py, line 12)

In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OrdinalEncoder
ord_enc = OrdinalEncoder()
from sklearn.model_selection import train_test_split

from sklearn.metrics import r2_score 
from sklearn.metrics import mean_squared_error

from datetime import date
from datetime import datetime

import math 

import warnings 
warnings.filterwarnings('ignore')
df_submit= pd.read_csv('sample_submission.csv') #load submission sample
df_train = pd.read_csv('train.csv') # load train dataset
df_test = pd.read_csv('test.csv') # load test dataset

df_test['type'] = "test"
df_train['type'] = "train"

df = pd.concat([df_test, df_train])

print('df_train:',df_train.shape)
print('df_test:',df_test.shape)
print('df:',df.shape)

Dataset provided are divided into training data (7000 observations) and test data (3000 observations) seperately. Training data consists 7000 entries and 61 variables, while test dataset contains 3000 observations and 60 variables and the target variable is proce that we will predict later. 